# 03 — Modelado Predictivo V2

**Objetivo:** Entrenar el modelo ganador utilizando el pipeline simplificado de 11 features.

**Diferencias V1 vs V2:**
- **V1:** Random Forest con 48 features (One-Hot). Propenso a ruido en categorías pequeñas.
- **V2:** Modelos basados en árboles sobre features Target-Encoded. Mayor estabilidad y velocidad.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay

X_train = pd.read_csv("../data/processed/X_train_v2.csv")
X_test = pd.read_csv("../data/processed/X_test_v2.csv")
y_train = pd.read_csv("../data/processed/y_train_v2.csv")["target"]
y_test = pd.read_csv("../data/processed/y_test_v2.csv")["target"]

## 1. Entrenamiento y Comparativa de Modelos

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    results.append({"Modelo": name, "ROC-AUC": auc})

df_res = pd.DataFrame(results)
print(df_res.sort_values(by="ROC-AUC", ascending=False))

## 2. Curvas ROC Comparativas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
plt.title("Comparativa de Curvas ROC (V2)")
plt.show()

## 3. Matrices de Confusión Individuales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i])
    axes[i].set_title(f"CM: {name}")
plt.show()

In [ ]:
# Guardar el mejor modelo
import joblib
joblib.dump(models["Random Forest"], "../models/best_model.joblib")